In [ ]:
import numpy as onp
from pathlib import Path
import sys
_project_root: Path = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
import src.symmetry as sym
from src.categorization import show_latex_bmatrix, show_latex_eqn, latex_bmatrix_vec, cyclic_power_rep
from src.bases import CanonBasis4x4
NDArray = onp.ndarray



<b>Exercise 4.2.</b> Here we study an induced representation on spaces of linear transformations.
(a) Consider the representations $\sigma, \tau : \mathbb{Z}/4\mathbb{Z} \to GL_2(C)$ given by
$$
\sigma(1) = \begin{bmatrix}  i & 0 \\ 0 & −1 \end{bmatrix} 
$$
and
$$
\tau(1) = \begin{bmatrix} 0 & −1 \\  1 & 0  \end{bmatrix}.
$$
These representations induce a $\mathbb{Z}/4\mathbb{Z}$-action on $\mathrm{Hom}(\mathbb{C}_\sigma^2, \mathbb{C}_\tau^2)$ via
$$
g \cdot L := \tau(g) \circ L \circ \sigma(g^{−1}).
$$
If 
$L = \begin{bmatrix} \alpha & \beta \\ \gamma & \delta \end{bmatrix} \in \mathrm{Hom}(\mathbb{C}_\sigma^2, \mathbb{C}_\tau^2)$, 
compute $1 \cdot L$. Use this to write the action of $1$ on $\mathrm{Hom}(\mathbb{C}_\sigma^2, \mathbb{C}_\tau^2)$ as a $4 \times 4$ matrix using the ordered basis 
$$
(E_{11}, E_{12}, E_{21}, E_{22}),
$$
where $E_{ij}$ is the matrix with $1$ in the $i$-th row and $j$-th column and $0$ elsewhere

(b) Find a basis for the space of invariants $\mathrm{Hom}(\mathbb{C}_\sigma^2, \mathbb{C}_\tau^2)^{\mathbb{Z}/4\mathbb{Z}}$. 
Hint: Recall Exercise 3.5.
Conclude that $\mathbb{C}_\sigma^2$ and $\mathbb{C}_\tau^2$, each of which decompose into two subrepresentations, share one irreducible subrepresentation in common.

For reference:

<b>Exercise 3.5.</b> Let $\rho: G \to GL(V)$ be a representation. Consider the Reynolds operator,
$$
\phi:=\frac{1}{|G|}\sum_{h\in G}\rho(h),
$$
given by averaging over the linear maps in $\mathrm{im}(G)$.

(a) Show that $\phi$ is a projection onto the fixed subspace $V^G$.

(b) Show that $\phi$ is an orthogonal projection with respect to $\langle\cdot ,\cdot\rangle_G$.

In [ ]:

B = CanonBasis4x4()

sigma1 = 1j * B.E11 - B.E22
tau1 = - B.E12 + B.E21
sigma1_inv = onp.linalg.inv(sigma1)

A = onp.column_stack([
    (tau1 @ E @ sigma1_inv).reshape(-1)
    for E in B
])

show_latex_bmatrix(A, name="A")
show_latex_bmatrix(A - onp.identity(4), name=r"A-\mathbb{I}")


rep = cyclic_power_rep(A, 4)

chi = rep.character()
P = rep.reynolds_projector()
print("character =", chi)
show_latex_bmatrix(P, name="P")

# 
u, s, vh = onp.linalg.svd(P)
inv_basis = vh.conj().T[:, s > 1e-10]

print("dim invariants =", inv_basis.shape[1])

v = inv_basis[:, 0]
v = v / v[onp.argmax(onp.abs(v))]   # normalize by first nonzero entry
show_latex_eqn(fr"\text{{vector in basis }}\mathcal{{B}}:=({','.join(B.names_latex())}) = {latex_bmatrix_vec(v)}")

L = v.reshape(2, 2)
show_latex_bmatrix(L, name="L")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

character = [4.+0.j 0.+0.j 0.+0.j 0.+0.j]


<IPython.core.display.Math object>

dim invariants = 1


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<b>Exercise 4.3.</b> In this problem we will compute some invariant polynomials. We urge the use of SageMath or other comparable tools for these calculations!

Consider the representation $A_4 \circlearrowright V := \mathbb{R}^3$ given in terms of the tetrahedral symmetries (see Example 3.50 and Example 5.4):
$$
(1 2 3) \mapsto \begin{bmatrix} 0 & 0 & 1 \\ 1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix}
\quad\text{and}\quad 
(1 2)(3 4) \mapsto \begin{bmatrix} -1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{bmatrix}.
$$
Since all matrices in this representation are orthogonal, $V$ is isomorphic to its dual representation $V^*$.

If we write $x, y, z$ for the usual coordinate functions on \mathbb{R}^3, we can think of $V^* =\text{Span}\{x, y, z\}$: each of these coordinate functions is a linear functional on V! Indeed, we can study the symmetric powers of $V^*$ to make sense of non-linear polynomial functions on $\mathbb{R}^3$. For example, we might consider
$$
\text{Sym}^2(V^*) = \text{Span}\{x^2, xy, y^2, xz, yz, z^2\}.
$$
More generally, $\text{Sym}^d(V∗)$ consists of the formal span of all degree $d$ monomials:
$$
\text{Sym}^d(V^∗) = \text{Span}\{x^a y^b z^c | a, b, c \in \mathbb{N} and a + b + c = d\}.
$$
There is a natural action of $G$ via $(g\cdot f)(v) = f(g^{−1} \cdot v)$ for all $v \in V$.
<!-- To facilitate the computations required in this problem, we offer Listing 4.1 which implements the above group action. This block of code creates a polynomial ring (over Q, which is all we need here) for the formal variables x, y, z; generates a group from the image of our generators and stores a list of all the resulting matrices in group_elements; and provides the act() function to compute what a given group element does to a particular polynomial. -->

Given a representation $G \circlearrowright V$, a polynomial 
$f \in \text{Sym}^d(V^∗)$ is 
$G$-<b>invariant</b> if 
$f \in \text{Sym}^d(V^∗)^G$.

(a) Let 
$f \in \text{Sym}^d(V^∗)^G$ and $c \in \mathbb{R}$. Show that 
$G \circlearrowright V$ restricts to an action on the level surface
$$
f^{−1}(\{c\}) := \{v \in \mathbb{R}^3 ∶ f(v) = c\}.
$$

(b) Describe the space of all $A_4$-invariant polynomials for $d \in\{1,\dots , 4\}$.
Hint: Use Exercise 3.5. There is no need to include your code, but you should indicate what calculations you carried out!

(c) Using your favorite graphing software, plot some level surfaces of the polynomials from part (b).
What sorts of surfaces do you find? Do they seem to enjoy any relevant symmetries?

In [ ]:
import plotly.graph_objects as go
from IPython.display import IFrame, display

g123 = onp.array([
    [0, 0, 1],
    [1, 0, 0],
    [0, 1, 0],
], dtype=int)

g12_34 = onp.array([
    [-1, 0, 0],
    [0, -1, 0],
    [0, 0, 1],
], dtype=int)

A4 = sym.generated_group(
  [
    sym.AffineOperation(g123, label="(123)"),
    sym.AffineOperation(g12_34, label="(12)(34)"),
  ],
  name="A4",
  max_order=12,
)

print("group order =", A4.order)

# Centered grid for implicit level-surface plots.
L = 1.0
N = 48
xi = L * onp.linspace(-1, 1, N)
grid = onp.meshgrid(xi, xi, xi, indexing="ij")
X, Y, Z = grid

def reynolds_field(func):
  return sym.symmetrize(func, *grid, sym_ops=A4)

# Reynolds averages of seed monomials. These recover the low-degree A4 invariants.
q2 = 3.0 * reynolds_field(lambda x, y, z: x**2)
q3 = reynolds_field(lambda x, y, z: x * y * z)
q4a = 3.0 * reynolds_field(lambda x, y, z: x**4)
q4b = 3.0 * reynolds_field(lambda x, y, z: x**2 * y**2)
quartic_combo = q4a - 2.0 * q4b

out_dir = Path("iframe_figures")
out_dir.mkdir(exist_ok=True)

def plot_level_surface(values, level, *, title="", opacity=0.6, colorscale="Viridis", filename="figure_0.html"):
  fig = go.Figure(data=go.Isosurface(
    x=X.ravel(),
    y=Y.ravel(),
    z=Z.ravel(),
    value=values.ravel(),
    isomin=level,
    isomax=level,
    surface_count=1,
    opacity=opacity,
    colorscale=colorscale,
    caps=dict(x_show=False, y_show=False, z_show=False),
  ))
  fig.update_layout(
    title=title,
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="cube",
    ),
    margin=dict(l=0, r=0, b=0, t=40),   
  )
  path = out_dir / filename
  fig.write_html(path, include_plotlyjs="cdn")
  display(IFrame(src=str(path), width="100%", height=700))
  return path

print("degree 2 invariant: x^2 + y^2 + z^2")
print("degree 3 invariant: xyz")
print("degree 4 invariants: x^4 + y^4 + z^4 and x^2 y^2 + x^2 z^2 + y^2 z^2")
print("Run one plot at a time to keep VS Code responsive.")

# Example calls for part (c):
# plot_level_surface(q2, 0.6, title=r"$x^2+y^2+z^2 = 0.6$", colorscale="Cividis", filename="sphere.html")
# plot_level_surface(q3, 0.08, title=r"$xyz = 0.08$", colorscale="Plasma", filename="xyz_pos.html")
# plot_level_surface(q3, -0.08, title=r"$xyz = -0.08$", colorscale="Plasma", filename="xyz_neg.html")
# plot_level_surface(q4a, 0.5, title=r"$x^4+y^4+z^4 = 0.5$", colorscale="Viridis", filename="quartic_sum.html")
# plot_level_surface(q4b, 0.15, title=r"$x^2 y^2 + x^2 z^2 + y^2 z^2 = 0.15$", colorscale="Magma", filename="quartic_pairwise.html")
# plot_level_surface(quartic_combo, 0.15, title=r"$x^4+y^4+z^4-2(x^2y^2+x^2z^2+y^2z^2)=0.15$", colorscale="Turbo", filename="quartic_combo.html")


<b>Exercise 4.4.</b> Consider the following operators on $V = \mathbb{C}^3$ and $W = \mathbb{C}^2$, respectively:
$$
A =  \begin{bmatrix} -1 & 0 & 0 \\ 0 & 0 & 1 \\ 0 & 1 & 0 \end{bmatrix}
\quad\text{and}\quad 
B = \begin{bmatrix} \alpha & \beta \\ \gamma & \delta \end{bmatrix}.
$$
Write down a basis for $V\otimes W$, use it to express $A\otimes B$ as a matrix, then verify that 
$\text{Tr}(A\otimes B) = \text{Tr}(A) \text{Tr}(B)$.

<b>Exercise 4.5.</b> (Bonus) Compute the eigenvalues of
$$
A =  \begin{bmatrix} 0 & 0 & -1 \\ 1 & 0 & -1 \\ 0 & 1 & -1 \end{bmatrix}
$$
without calculating any determinants.
<i>Hint:</i> $A^4=\mathbb{I}$

<b>Exercise 5.1.</b> Recall that $\mathcal{S}_4$ encodes the orientation-preserving isometries of the cube; there is one such
rotation for each permutation of the 4 diagonals of the cube. In particular, $\mathcal{S}_4$ acts (a) on the 8 vertices of
the cube and (b) on the 12 edges of the cube.

a) Decompose the permutation representation of $\mathcal{S}_4$ on the vertices of the cube into irreducibles.

b) Decompose the permutation representation of $\mathcal{S}_4$ on the edges of the cube into irreducibles.

<b>Exercise 5.2.</b> The abelianization of $\mathcal{D}_8$
is $\mathcal{D}_8/\langle r^2\rangle\cong \mathcal{V}$. Compute the character table of $\mathcal{D}_8$.

Exercise 5.3. In this problem we compute the character table of $\mathcal{S}_5$. We start with only knowledge of the conjugacy classes and their sizes:
$$
wtf
$$

a) Let $U$, $U'$, and $V$ be the trivial, alternating, and standard representations of 
$\mathcal{S}_5$, respectively. Find the characters of these representations and confirm that they are irreducible.

b) Set $V' = V \otimes U'$, find its character, and confirm that it is a distinct irreducible representation from $U$, $U'$, and $V$.

c) If $Y$ is a representation of $G$, then we can compute the characters of its symmetric powers $\mathrm{Sym}^k(Y)$ (c.f. Exercise 4.3) in terms of $\chi_Y$ . In particular, you may use the following formula without proof:
$$
\chi_{\mathrm{Sym}^k(Y)}(g) = \frac{1}{2}(\chi_Y (g)^2 + \chi_Y(g^2)).
$$
Compute the character for $\mathrm{Sym}^2(V)$ (with $V$ the standard representation) and use this to show that 
$\mathrm{Sym}^2(V)$ is a direct sum of three distinct irreducible representations.

d) Show that $\mathrm{Sym}^2(V)$ is the direct sum of $U$, $V$, and one previously unknown irreducible—call it $W$.

e) Complete the character table for $\mathcal{S}_5$


<b>Exercise 5.4.</b> Consider the class function $f: \mathcal{S}_5 \to \mathbb{C}$, where $f(\sigma)$ counts the number of 2-cycles in the cycle notation for $\sigma$, e.g.:
$$
f((2 3 4)) = 0,\\
f((1 5)) = 1,\\
f((1 2)(3 4)) = 2.
$$
Write $f$ in the basis of irreducible characters for $\mathcal{C}(\mathcal{S}_5)$.

<b>Exercise 5.5.</b> If $\rho∶ G \to GL(V)$ is faithful (i.e., that $\text{ker}\rho=\{e\}$), show that every irreducible representation of $G$ is contained in some tensor power $V^{\otimes k}$.

<i>Hint:</i> If $W$ is an irreducible representation, consider the generating function
$$
\sum_{k=0}^{\infty}\langle \chi_W, \chi_{V^{\otimes k}}\rangle t^k
$$
If you can show this series converges to a nontrivial rational function, it must contain
(infinitely many) non-zero coefficients!

Let $\rho: G\to GL(V)$ be an irreducible representation. If $\tilde{G}\twoheadrightarrow G$ is a surjective homomorphism, then the <b>pullback representation</b>
$$
\tilde{G}\twoheadrightarrow G\underrightarrow{\rho} GL(V)
$$ 
is also irreducible

Suppose that $G$ has non-isomorphic irreducible representations $V_1$, $V_2$, and $V_3$. Suppose further that $V$ and $W$ are $G$-representations which decompose as 
$$
V=V_1^{\oplus 3}\oplus V_2\quad\text{and}\quad W=V_1\oplus V_2^{\oplus 2}\oplus V_3.
$$
What is the dimension of the space of intertwiners $\mathrm{Hom}_G(V,W)$?

<!-- Im assuming the dimension has to be 0 purely by Schur's lemma, as we can pair up $V_1\oplus V_2$ in both $V$ and $W$ and then the problem becomes relating 
$$
V_1^{\oplus 2}\quad\text{and}\quad V_2\oplus V_3,
$$
but $V_1$ could ahve dimension 2 and $V_2$, $V_3$ have dimensions 1 and 3 so im not sure.  -->

In this problem, you may use the fact that the abelianization of 
$$
\mathcal{A}_4 = \langle (1 2 3), (1 2)(3 4)\rangle
$$
is cyclic of order 3. This relationship can be expressed via the homomorphism 
$$
\mathcal{A}_4\twoheadrightarrow \mathbb{Z}/3\mathbb{Z}\\
(1 2 3)\mapsto 1 \\
(1 2)(3 4)\mapsto 0.
$$
(a) Explain why $\mathbb{Z}/3\mathbb{Z}$ has at most 3 irreducible representations, and then write out three distinct irreducible representations of $\mathbb{Z}/3\mathbb{Z}$.

<i>Remark</i>: You may cite any materials in the notes up through Theorem 5.3, but you <b>may not</b> appeal to character theory.

(b) Find all 1-dimensional irreducible representations of $\mathcal{A}_4$.

<i>Remark</i>: For full credit, you should prove that you have found them all!

Here you may use the fact that every irreducible representation of $S_3$ is isomorphic to exactly one of the following:
- the trivial rep $U$
- the alternating rep $U'$
- the standard rep $V$
Consider the permutation representation $S_3$ acting on $\mathbb{C}^3$. Manually (i.e., WITHOUT THE TOOLS OF CHARACTER THEORY), decompse $\mathrm{Sym}^3(\mathbb{C}^3)$ (c.f., Exercise 4.3) into isotypic components:
$$
\mathrm{Sym}^3(\mathbb{C}^3) = U^{\oplus a_1}\oplus (U')^{\oplus a_2} \oplus V^{\oplus a_3}
$$
<i>Remark</i>: For full credit, write out a <i>basis</i> for each isotypic component. Partial credit includes:
- Correctly determining the multiplicity of each $U$, $U'$, and $V$ in $\mathrm{Sym}^3(\mathbb{C}^3)$.
- Finding a basis for $\mathrm{Sym}^3(\mathbb{C}^3)^{S_3}$ (i.e., identifying the isotypic component of $U$).
- Identifying bases for subrepresentations, even if you are unable to fully decompose.


In [4]:
import itertools as it
import numpy as onp
import src.symmetry as sym


mons = [
    (3,0,0), (0,3,0), (0,0,3),
    (2,1,0), (2,0,1), (1,2,0), (0,2,1), (1,0,2), (0,1,2),
    (1,1,1),
]
mon_names = [
    "x1^3", "x2^3", "x3^3",
    "x1^2x2", "x1^2x3", "x2^2x1", "x2^2x3", "x3^2x1", "x3^2x2",
    "x1x2x3",
]
idx = {m: i for i, m in enumerate(mons)}

def sym3_matrix(perm):
  # perm is a tuple like (1,0,2) for swapping x1,x2
  A = onp.zeros((10, 10), dtype=int)
  for j, alpha in enumerate(mons):
    beta = [0, 0, 0]
    for exp, dst in zip(alpha, perm):
      beta[dst] += exp
    A[idx[tuple(beta)], j] = 1
  return A

S3 = list(it.permutations(range(3)))
rho = {g: sym3_matrix(g) for g in S3}
print(rho)

rep = sym.Representation(group=S3, matrices=rho, name="Sym^3(C^3)")
P = rep.reynolds_projector()

print("rank(P) =", onp.linalg.matrix_rank(P))
print(onp.real_if_close(P))


{(0, 1, 2): array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]]), (0, 2, 1): array([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]]), (1, 0, 2): array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 